# Autogen + Zep long-term memory

Builds a simple Autogen conversable agent that persists turns to Zep threads and
injects `thread.get_user_context` into the system prompt.

Requires `ZEP_API_KEY` and `OPENAI_API_KEY`.


In [ ]:
# %pip install -q pyautogen zep-cloud python-dotenv


In [ ]:
import asyncio
import os
import uuid
from typing import Optional

from autogen import ConversableAgent
from dotenv import load_dotenv
from zep_cloud.client import AsyncZep
from zep_cloud.types import Message

load_dotenv()
assert os.environ.get("ZEP_API_KEY"), "ZEP_API_KEY is required"
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY is required"

zep = AsyncZep(api_key=os.environ["ZEP_API_KEY"])
llm_config = {"config_list": [{"model": "gpt-4o-mini", "api_key": os.environ["OPENAI_API_KEY"]}]}


In [ ]:
class ZepConversableAgent(ConversableAgent):
    """ConversableAgent that reads/writes Zep thread memory each turn."""

    def __init__(self, name, zep_client, zep_thread_id, zep_user_name, **kwargs):
        super().__init__(name=name, **kwargs)
        self.zep_client = zep_client
        self.zep_thread_id = zep_thread_id
        self.zep_user_name = zep_user_name
        self._base_system = kwargs.get("system_message", "")

    async def a_generate_reply(self, messages=None, sender=None, **kwargs):
        messages = messages or self.chat_messages[sender]
        last = messages[-1]
        await self.zep_client.thread.add_messages(
            thread_id=self.zep_thread_id,
            messages=[
                Message(
                    role="user",
                    name=self.zep_user_name,
                    content=last.get("content", ""),
                )
            ],
        )
        memory = await self.zep_client.thread.get_user_context(thread_id=self.zep_thread_id)
        context = memory.context or ""
        self.update_system_message(
            self._base_system
            + f"\n\nRelevant long-term memory from Zep:\n{context}"
        )
        reply = await super().a_generate_reply(messages=messages, sender=sender, **kwargs)
        if reply:
            await self.zep_client.thread.add_messages(
                thread_id=self.zep_thread_id,
                messages=[Message(role="assistant", name=self.name, content=str(reply))],
            )
        return reply


In [ ]:
user_name = "Cathy"
user_id = user_name + uuid.uuid4().hex[:4]
thread_id = str(uuid.uuid4())

await zep.user.add(user_id=user_id, first_name=user_name, email=f"{user_id}@example.com")
await zep.thread.create(thread_id=thread_id, user_id=user_id)

prior = [
    Message(role="assistant", name="CareBot", content=f"Hi {user_name}, how are you feeling today?"),
    Message(role="user", name=user_name, content="I've been grieving my mother. Some days are heavy."),
    Message(role="assistant", name="CareBot", content="I'm sorry for your loss. What usually helps on hard days?"),
    Message(role="user", name=user_name, content="Short walks and looking through old photos."),
]
await zep.thread.add_messages(thread_id=thread_id, messages=prior)
print({"user_id": user_id, "thread_id": thread_id})


In [ ]:
carebot = ZepConversableAgent(
    name="CareBot",
    zep_client=zep,
    zep_thread_id=thread_id,
    zep_user_name=user_name,
    system_message="You are a compassionate caregiver. Use Zep memory when helpful.",
    llm_config=llm_config,
    human_input_mode="NEVER",
)
cathy = ConversableAgent(
    name=user_name,
    system_message="You are Cathy. Keep replies brief.",
    llm_config=llm_config,
    human_input_mode="NEVER",
)

# Keep the demo short for non-interactive runs
chat_result = await carebot.a_initiate_chat(
    cathy,
    message="Hi Cathy, nice to see you again. How are you holding up?",
    max_turns=2,
)
print(chat_result)
context = await zep.thread.get_user_context(thread_id=thread_id)
print(context.context)
search = await zep.graph.search(user_id=user_id, query="family", limit=3, scope="edges")
print(getattr(search, "edges", search))
